# Phase 1: 256x256 Conditional DCGAN - Kaggle GPU Training

**Project:** GAN-Based Synthetic Medical Image Augmentation for COVID-19 Chest X-ray Classification
**Phase:** 1 — Higher resolution (256x256) Conditional DCGAN

Trains a 256x256 Conditional DCGAN on Kaggle GPU, generates 100 synthetic images per class,
trains a ResNet-18 classifier, evaluates on the test set, and computes FID vs the 128x128 cDCGAN.

---

## 📋 Setup Instructions

1. **Enable GPU:** Settings → Accelerator → GPU (P100 or T4)
2. **Enable Internet:** Settings → Internet → On (needed for pip install)
3. **Add Dataset:** Add the COVID-19 Radiography Database from Kaggle's Add Input panel
   - Dataset slug example: `tawsifurrahman/covid19-radiography-database`
   - Update `KAGGLE_DATASET_SLUG` below if different

---


In [ ]:
# Check GPU availability (Kaggle provides P100 or T4)
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

In [ ]:
# Install missing non-torch packages (PyTorch is pre-installed on Kaggle)
!pip install scikit-learn scipy matplotlib seaborn pandas pillow tqdm -q

In [ ]:
# Clone the project repo (Kaggle Linux pattern, like LOWDATA_KAGGLE.ipynb)
!git clone https://github.com/nanthakishoren2005/Enhancing-Clinical-Data-with-GAN-Based-Synthetic-Data-Generation.git
%cd Enhancing-Clinical-Data-with-GAN-Based-Synthetic-Data-Generation
import os
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Link Kaggle-attached COVID-19 Radiography Database to archive/COVID-19_Radiography_Dataset
# Then run dataset preparation

import os
from pathlib import Path

# Update this slug to match your attached Kaggle dataset
KAGGLE_DATASET_SLUG = "tawsifurrahman/covid19-radiography-database"
kaggle_input_path = Path("/kaggle/input") / KAGGLE_DATASET_SLUG
archive_path = Path("archive") / "COVID-19_Radiography_Dataset"

print(f"Looking for dataset at: {kaggle_input_path}")
print(f"Dataset exists: {kaggle_input_path.exists()}")

if kaggle_input_path.exists():
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    if not archive_path.exists():
        os.symlink(str(kaggle_input_path), str(archive_path))
        print(f"Symlinked: {archive_path} -> {kaggle_input_path}")
    else:
        print(f"Archive path already exists: {archive_path}")
else:
    print("⚠️ Kaggle dataset not found at expected path.")
    print("   Please add the COVID-19 Radiography Database via Kaggle's Add Input panel.")
    print("   Then update KAGGLE_DATASET_SLUG above and rerun this cell.")

In [ ]:
# Run dataset preparation (partitions 600 train / 100 val / 100 test per class)
!python -m scripts.prepare_covid_dataset

In [ ]:
# Verify Phase 1 files exist and data is prepared
from pathlib import Path

phase1_files = [
    "src/gan/cdcgan256.py",
    "configs/gan256.yaml",
    "scripts/train_gan256.py",
    "scripts/__init__.py",
]

print("=== Phase 1 Files ===")
for f in phase1_files:
    exists = Path(f).exists()
    print(f"{'✅' if exists else '❌'} {f}")

print("\n=== Data Splits ===")
for split in ["train", "val", "test"]:
    split_dir = Path(f"data/original/{split}")
    if split_dir.exists():
        total = 0
        for cls in ["normal", "covid", "lung_opacity", "viral_pneumonia"]:
            cls_dir = split_dir / cls
            if cls_dir.exists():
                n = len([f for f in cls_dir.iterdir() if f.suffix.lower() in [".png", ".jpg", ".jpeg"]])
                total += n
                print(f"  {split}/{cls}: {n}")
        print(f"  {split} total: {total}")
    else:
        print(f"  {split}: NOT FOUND")

print("\n✅ All Phase 1 files present!" if all(Path(f).exists() for f in phase1_files) else "\n⚠️ Some files missing")

---
## 🚀 Phase 1A: Train 256x256 Conditional DCGAN (50 epochs)

⏱️ **Estimated time on Kaggle GPU:** 15-30 minutes

Note: `train_gan256.py` resizes images to 256x256 on load via `get_gan_transforms(img_size=256)`.
No separate 256 preparation step is needed.

CLI flags used (verified against `scripts/train_gan256.py`):
- `--epochs`, `--batch-size`, `--lr-g`, `--lr-d`, `--resume`
- `--skip-train`, `--skip-generate`, `--skip-classifier`, `--skip-fid`

---

In [ ]:
!python -m scripts.train_gan256 --epochs 50 --batch-size 16 --skip-classifier --skip-fid

---
## 🎨 Phase 1B: Generate 100 Synthetic Images per Class (256x256)

Saves to `outputs/v2/res256/synthetic/<class>/`.

---

In [ ]:
!python -m scripts.train_gan256 --skip-train --skip-classifier --skip-fid

---
## 🏥 Phase 1C: Train ResNet-18 Classifier on 256x256 Inputs

Identical protocol to baseline: ResNet18, 15 epochs max, patience 5.
Train images resized to 256x256 for this model only.

---

In [ ]:
!python -m scripts.train_gan256 --skip-train --skip-generate --skip-fid

---
## 📊 Phase 1D: Evaluate on Held-Out Test Set

Test images resized to 256x256 for this model only.
Reports accuracy/macro F1 vs baseline 92.75%.

---

In [ ]:
!python -m scripts.train_gan256 --skip-train --skip-generate --skip-classifier

---
## 📈 Phase 1E: Compute FID (256x256 cDCGAN vs 128x128 cDCGAN)

Uses InceptionV3 features for FID computation.

---

In [ ]:
!python -m scripts.train_gan256 --skip-train --skip-generate --skip-classifier

---
## 📁 Results

All outputs saved to: `outputs/v2/res256/`

Files:
- `cgan256_checkpoint.pt` - Trained GAN model
- `cgan256_loss_curve.png` - Training loss curve
- `cgan256_epoch_*.png` - Sample images per epoch
- `synthetic/` - 100 synthetic images per class
- `cnn256_best.pt` - Trained classifier
- `cnn256_augmented_learning_curve.png` - Classifier learning curve
- `phase1_summary.json` - Complete results summary
- `fid_comparison.json` - FID vs 128x128 cDCGAN

---
## 📥 Download Results

Zip results to /kaggle/working/ and download from Kaggle's Output panel.
Save Version to persist outputs.

---

In [ ]:
# Zip Phase 1 results to /kaggle/working/ for download
import zipfile
from pathlib import Path

output_dir = Path("outputs/v2/res256")
zip_path = Path("/kaggle/working/res256_results.zip")

if output_dir.exists():
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for file in output_dir.rglob("*"):
            zipf.write(file, file.relative_to(output_dir.parent))
    print(f"✅ Zipped results to: {zip_path}")
    print(f"   Zip size: {zip_path.stat().st_size / 1024 / 1024:.2f} MB")
    print("\n📥 Download from Kaggle's Output panel (right sidebar).")
    print("💾 Click 'Save Version' to persist outputs.")
else:
    print(f"⚠️ Output directory not found: {output_dir}")